## Obtaining Product Reviews

In [ ]:
import pandas as pd
df = pd.read_csv('reviews.csv')

In [ ]:
print('Number of reviews: ', df.count())

In [ ]:
df.head(3)

## Cleansing the data

In [ ]:
# Used a different translator that was able to work
from googletrans import Translator

In [ ]:
detector = Translator()
detection = detector.detect('Good')
print(detection)

In [ ]:
#pip install git+https://github.com/BoseCorp/py-googletrans.git --upgrade

## Removing Non-English Reviews

In [ ]:
# Added a timeout value due to the original run having a connection timeout
# code is a little different fro the book
translator = Translator(timeout=10) 
detector = Translator()
df['lang'] = df['title'].apply(lambda x: detector.detect(x).lang)

In [ ]:
print(df[['title','rating','lang']])

In [ ]:
df = df[df["lang"]=='en']

In [ ]:
df.head(5)

## Splitting and transforming the data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
reviews = df['title'].values
ratings = df['rating'].values
reviews_train, reviews_test, y_train, y_test = train_test_split(reviews,
                ratings, test_size=0.2, random_state=1000)
vectorizer=CountVectorizer()
vectorizer.fit(reviews_train)
x_train = vectorizer.transform(reviews_train)
x_test = vectorizer.transform(reviews_test)

In [ ]:
print(len(x_train.toarray()))

In [ ]:
print(len(x_test.toarray()))

In [ ]:
print(len(x_train.toarray()[0]))

In [ ]:
print(x_train.toarray())

## Training the model

In [ ]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression()
classifier.fit(x_train, y_train)

## Evaluating the model

In [ ]:
import numpy as np
predicted = classifier.predict(x_test)
accuracy1 = classifier.score(x_test, y_test)

In [ ]:
print(accuracy1)

In [ ]:
# Could have used the score value from the model
accuracy = np.mean(predicted == y_test)
print(accuracy)

In [ ]:
from sklearn import metrics
print(metrics.confusion_matrix(y_test, predicted, labels = [1,2,3,4,5]))

In [ ]:
print(df.groupby("rating").size())

In [ ]:
print(metrics.classification_report(y_test, predicted, labels=[1,2,3,4,5]))

## Predicting Stock trends

In [ ]:
import yfinance as yf
tkr = yf.Ticker("AAPL")
hist = tkr.history(period='1y')


In [ ]:
# if the date has minutes and other pieces, run this to get to a basic date
hist.index = hist.index.tz_localize(None)

In [ ]:
import pandas_datareader.data as pdr
from datetime import date, timedelta

In [ ]:
end = date.today()
start = end - timedelta(days=365)
index_data = pdr.get_data_stooq('^SPX', start, end)

In [ ]:
hist.head(3)

In [ ]:
index_data.head(3)

In [ ]:
df = hist.join(index_data, rsuffix='_idx')

In [ ]:
df.head()

In [ ]:
df = df[['Close','Volume','Close_idx','Volume_idx']]

In [ ]:
df.head(3)

## Deriving Features from Continuous Data

In [ ]:
df['priceRise'] = np.log(df['Close'] / df['Close'].shift(1))
df['volumeRise'] = np.log(df['Volume'] / df['Volume'].shift(1))
df['priceRise_idx'] = np.log(df['Close_idx'] / df['Close_idx'].shift(1))
df['volumeRise_idx'] = np.log(df['Volume_idx'] / df['Volume_idx'].shift(1))
df = df.dropna()

In [ ]:
df = df[['priceRise','volumeRise','priceRise_idx','volumeRise_idx']]

In [ ]:
df.head(3)

## Generating the output variable

In [ ]:
conditions = [
(df['priceRise'].shift(-1) > 0.01),
(df['priceRise'].shift(-1)< -0.01)
]
choices = [1,-1]
df['Pred'] = np.select(conditions, choices, default=0)

In [ ]:
df.head(3)

## Training and evaluating the model

In [ ]:
features = df[['priceRise','volumeRise','priceRise_idx','volumeRise_idx']].to_numpy()
features = np.around(features, decimals=2)
target = df['Pred'].to_numpy()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

rows_train, rows_test, y_train, y_test = train_test_split(features, target, test_size=0.2)
clf = LogisticRegression()
clf.fit(rows_train, y_train)

In [ ]:
print(clf.score(rows_test, y_test))